In [18]:
# ==================================================
# IMPORTS
# ==================================================

from pathlib import Path

import warnings
import time
import random

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

In [19]:
# ==================================================
# CONFIG
# ==================================================

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

EXPORT_FOLDER = Path("standardized_exports")
RESULTS_FOLDER = Path("results")

RESULTS_FOLDER.mkdir(exist_ok=True)

print("Export Folder :", EXPORT_FOLDER)
print("Results Folder:", RESULTS_FOLDER)

Export Folder : standardized_exports
Results Folder: results


In [ ]:
HORIZON_DICT = {

    # Traffic (4 hours)
    "traffic_small": 48,
    "traffic_medium": 48,
    "traffic_large": 48,

    # Energy
    "energy_small": 24,
    "energy_medium": 24,
    "energy_large": 24,

    # Favorita (28 days)
    "favorita-train_small": 28,
    "favorita-train_medium": 28,
    "favorita-train_large": 28,

    "favorita-transaction_small": 28,
    "favorita-transaction_medium": 28,
    "favorita-transaction_large": 28,

    # Agricultural (5 years)
    "agri_small": 5,
    "agri_medium": 5,
    "agri_large": 5,

    # M4 Finance
    "m4-daily_small": 20,
    "m4-daily_medium": 20,
    "m4-daily_large": 20,

    "m4-hourly_small": 20,
    "m4-hourly_medium": 20,
    "m4-hourly_large": 20,

    "m4-weekly_small": 20,
    "m4-weekly_medium": 20,
    "m4-weekly_large": 20,

    "m4-monthly_small": 20,
    "m4-monthly_medium": 20,
    "m4-monthly_large": 20,

    "m4-quarterly_small": 20,
    "m4-quarterly_medium": 20,
    "m4-quarterly_large": 20,

    "m4-yearly_small": 20,
    "m4-yearly_medium": 20,
    "m4-yearly_large": 20,
    }

In [42]:
print(HORIZON_DICT["energy_small"])


672


In [21]:
# ==================================================
# DISCOVER DATASETS
# ==================================================

AVAILABLE_DATASETS = sorted(
    [file.stem for file in EXPORT_FOLDER.glob("*.csv")]
)

print(f"Datasets Found: {len(AVAILABLE_DATASETS)}")

for name in AVAILABLE_DATASETS:
    print(name)

Datasets Found: 22
agri_medium
agri_small
energy_medium
energy_small
favorita-train_medium
favorita-train_medium_cov
favorita-train_small
favorita-train_small_cov
favorita-transaction_medium
favorita-transaction_small
m4-daily_medium
m4-daily_small
m4-hourly_medium
m4-hourly_small
m4-quarterly_medium
m4-quarterly_small
m4-weekly_medium
m4-weekly_small
m4-yearly_medium
m4-yearly_small
traffic_medium
traffic_small


In [22]:
# ==================================================
# SMAPE
# ==================================================

def smape(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    denominator = (
        np.abs(y_true) +
        np.abs(y_pred)
    )

    denominator = np.where(
        denominator == 0,
        1,
        denominator
    )

    return np.mean(
        200 *
        np.abs(y_true - y_pred) /
        denominator
    )

In [23]:
# ==================================================
# METRICS
# ==================================================

def calculate_metrics(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = np.mean(
        np.abs(y_true - y_pred)
    )

    rmse = np.sqrt(
        np.mean(
            (y_true - y_pred) ** 2
        )
    )

    smape_score = smape(
        y_true,
        y_pred
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "SMAPE": smape_score
    }

In [24]:
# ==================================================
# LOAD DATASET
# ==================================================

def load_dataset(dataset_name):

    path = EXPORT_FOLDER / f"{dataset_name}.csv"

    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(path)

    df["ds"] = pd.to_datetime(df["ds"])

    df = df.sort_values(
        ["unique_id", "ds"]
    )

    df = df.reset_index(drop=True)

    allowed_cols = [
    "unique_id",
    "ds",
    "y",
]

    df = df[
        [c for c in df.columns if c in allowed_cols]
    ]   

    return df

In [25]:
# ==================================================
# DATA CLEANING
# ==================================================

def clean_dataset(df):

    df = df.copy()

    df["y"] = pd.to_numeric(
        df["y"],
        errors="coerce"
    )

    df = df.replace(
        [np.inf, -np.inf],
        np.nan
    )

    df["y"] = (
        df.groupby("unique_id")["y"]
        .transform(
            lambda x:
            x.ffill().bfill()
        )
    )

    df = df.dropna(
        subset=["y"]
    )

    return df.reset_index(drop=True)

In [26]:
# ==================================================
# DATASET SUMMARY
# ==================================================

def dataset_summary(df):

    return pd.DataFrame({
        "Rows": [len(df)],
        "Series": [
            df["unique_id"].nunique()
        ],
        "Start": [
            df["ds"].min()
        ],
        "End": [
            df["ds"].max()
        ]
    })

In [27]:
# ==================================================
# TRAIN TEST SPLIT
# ==================================================

def split_dataset(
    df,
    horizon
):

    train_parts = []
    test_parts = []

    for uid, group in df.groupby("unique_id"):

        group = group.sort_values("ds")

        if len(group) <= horizon:
            continue

        train_parts.append(
            group.iloc[:-horizon]
        )

        test_parts.append(
            group.iloc[-horizon:]
        )

    print(horizon)


    train_df = pd.concat(
        train_parts,
        ignore_index=True
    )

    test_df = pd.concat(
        test_parts,
        ignore_index=True
    )

    return train_df, test_df

In [28]:
# ==================================================
# HORIZON LOOKUP
# ==================================================

def get_horizon(dataset_name):

    if dataset_name not in HORIZON_DICT:
        raise ValueError(
            f"No horizon configured for {dataset_name}"
        )

    return HORIZON_DICT[dataset_name]

In [29]:
# ==================================================
# LOAD AND PREPARE
# ==================================================

def prepare_dataset(dataset_name):

    df = load_dataset(dataset_name)

    df = clean_dataset(df)

    horizon = get_horizon(
        dataset_name
    )

    if "onpromotion" in df.columns:
        df["onpromotion"] = (
            df["onpromotion"]
            .astype(str)
            .str.lower()
            .map({
                "true": 1,
                "false": 0
            })
            .fillna(0)
            .astype(int)
        )

    train_df, test_df = split_dataset(
        df,
        horizon
    )
    

    return (
        df,
        train_df,
        test_df,
        horizon
    )

In [30]:
# ==================================================
# CREATE RESULT FILE
# ==================================================

def initialize_results_file(
    result_file
):

    result_path = (
        RESULTS_FOLDER /
        result_file
    )

    if not result_path.exists():

        columns = [

            "model",
            "dataset",
            "n_series",
            "n_rows",
            "horizon",
            "runtime_seconds",

            "MAE",
            "RMSE",
            "SMAPE"
        ]

        pd.DataFrame(
            columns=columns
        ).to_csv(
            result_path,
            index=False
        )

    return result_path

In [31]:
# ==================================================
# APPEND RESULT
# ==================================================

def append_result(
    result_file,
    result_dict
):

    result_path = (
        RESULTS_FOLDER /
        result_file
    )

    existing = pd.read_csv(
        result_path
    )

    existing = pd.concat(
        [
            existing,
            pd.DataFrame(
                [result_dict]
            )
        ],
        ignore_index=True
    )

    existing.to_csv(
        result_path,
        index=False
    )

In [32]:
# ==================================================
# SKIP COMPLETED RUNS
# ==================================================

def get_completed_runs(
    result_file
):

    result_path = (
        RESULTS_FOLDER /
        result_file
    )

    if not result_path.exists():
        return set()

    df = pd.read_csv(
        result_path
    )

    return set(
        zip(
            df["dataset"],
            df["model"]
        )
    )

In [33]:
# ==================================================
# TIMER
# ==================================================

def start_timer():

    return time.time()


def stop_timer(start):

    return (
        time.time() - start
    )

In [34]:
for dataset in AVAILABLE_DATASETS:

    try:

        df, train_df, test_df, horizon = (
            prepare_dataset(dataset)
        )

        print(
            f"{dataset:<25}"
            f" rows={len(df):<10}"
            f" series={df['unique_id'].nunique():<8}"
            f" horizon={horizon}"
        )

    except Exception as e:

        print(
            f"{dataset} -> ERROR: {e}"
        )

5
agri_medium               rows=15343      series=246      horizon=5
5
agri_small                rows=14428      series=246      horizon=5
672
energy_medium -> ERROR: No objects to concatenate
672
energy_small -> ERROR: No objects to concatenate
28
favorita-train_medium     rows=1985155    series=4504     horizon=28
favorita-train_medium_cov -> ERROR: No horizon configured for favorita-train_medium_cov
28
favorita-train_small      rows=609961     series=4504     horizon=28
favorita-train_small_cov -> ERROR: No horizon configured for favorita-train_small_cov
28
favorita-transaction_medium rows=1120       series=2        horizon=28
28
favorita-transaction_small rows=280        series=2        horizon=28
20
m4-daily_medium           rows=1129574    series=4227     horizon=20
20
m4-daily_small            rows=295890     series=4227     horizon=20
20
m4-hourly_medium          rows=353500     series=414      horizon=20
20
m4-hourly_small           rows=99360      series=414      horizon=20


KeyboardInterrupt: 